In [ ]:
# ============================================================================
# Is every pose actually IN the pocket?
#
# A 10 A ligand RMSD after CA superposition is larger than the CYP3A4 active
# site, so those outliers may be surface poses rather than real alternatives
# -- which would mean the "scattered" compounds have less usable diversity
# than the spread suggests. Centroid-to-Fe distance separates the two cases.
# ============================================================================
import numpy as np, pandas as pd, gemmi
from pathlib import Path

POSE_DIR  = Path("PUT_YOUR_CIF_OR_PDB_DIR_HERE")
PATTERN   = "*_model.cif"          # or "*_model.pdb"
PROT_CHAIN, HEME_CHAIN, LIG_CHAIN = "A", "B", "C"
IN_POCKET_MAX = 14.0               # A, centroid-to-Fe; generous for CYP3A4

rows = []
for p in sorted(POSE_DIR.glob(PATTERN)):
    if p.stem.endswith("_full"):
        continue
    try:
        st = gemmi.read_structure(str(p)); st.setup_entities()
        fe, lig, prot = None, [], []
        for ch in st[0]:
            for r in ch:
                for a in r:
                    if a.element.name.upper() == "FE":
                        fe = np.array(a.pos.tolist())
                if ch.name == LIG_CHAIN:
                    lig += [a.pos.tolist() for a in r if a.element.name != "H"]
                elif ch.name == PROT_CHAIN:
                    prot += [a.pos.tolist() for a in r if a.element.name != "H"]
        if fe is None or not lig:
            rows.append({"pose": p.stem, "note": "no Fe or no ligand"}); continue
        L, P = np.array(lig), np.array(prot)
        cen = L.mean(0)
        d_fe = float(np.linalg.norm(cen - fe))
        near = float(np.linalg.norm(L - fe, axis=1).min())
        # how buried: protein heavy atoms within 4.5 A of any ligand atom
        dmat = np.linalg.norm(L[:, None, :] - P[None, :, :], axis=-1)
        contacts = int((dmat.min(0) < 4.5).sum())
        cmpd = p.stem.split("_seed_")[0]
        rows.append({"cmpd": cmpd, "pose": p.stem, "cen_to_Fe": d_fe,
                     "closest_to_Fe": near, "prot_contacts": contacts,
                     "in_pocket": d_fe <= IN_POCKET_MAX, "note": ""})
    except Exception as ex:
        rows.append({"pose": p.stem, "note": f"{type(ex).__name__}: {ex}"[:60]})

D = pd.DataFrame(rows)
ok = D[D.get("in_pocket").notna()] if "in_pocket" in D else D
print(f"{len(D)} poses, {int(D.in_pocket.sum())} within {IN_POCKET_MAX} A of Fe\n")
print(D.groupby("cmpd").agg(
        n=("pose", "count"),
        out_of_pocket=("in_pocket", lambda s: int((~s).sum())),
        cen_min=("cen_to_Fe", "min"), cen_max=("cen_to_Fe", "max"),
        contacts_min=("prot_contacts", "min"),
        contacts_med=("prot_contacts", "median"))
      .round(1).to_string())

stray = D[~D.in_pocket.fillna(True)]
if len(stray):
    print(f"\n[!] {len(stray)} pose(s) outside the pocket -- these are not "
          f"usable alternatives, so the real spread is narrower than the "
          f"RMSD table suggests:")
    print(stray[["pose", "cen_to_Fe", "prot_contacts"]].round(1)
            .to_string(index=False))
else:
    print("\nAll poses are in the pocket; the wide spreads are genuine "
          "alternative binding modes, not stray placements.")